# CNN-DAE — Validação do `best_model.keras`

Valida um modelo treinado sem rodar o treino novamente.

**Antes de começar:** Runtime → Change runtime type → T4 GPU.

**Arquivos necessários para upload (célula 4):**
- `best_model.keras`
- `data_io.py`
- `contaminate.py`
- `dataset.py`
- `model.py`

---
## 1 · Google Drive (opcional)
Só necessário para salvar os gráficos. Pode pular.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/cnn_dae'
os.makedirs(f'{DRIVE_DIR}/validacao', exist_ok=True)
print('Drive montado. Pasta de validação:', f'{DRIVE_DIR}/validacao')

---
## 2 · Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU encontrada:', result.stdout.strip())
else:
    print('AVISO: sem GPU. O predict ainda funciona na CPU, mas mais lento.')

---
## 3 · Instalar dependências

In [ ]:
!pip install -q wfdb
import tensorflow as tf, wfdb, numpy as np
print('TensorFlow :', tf.__version__)
print('wfdb       :', wfdb.__version__)
print('numpy      :', np.__version__)
print('GPUs visíveis:', tf.config.list_physical_devices('GPU'))

---
## 4 · Upload dos arquivos
Envie os **5 arquivos**: `best_model.keras`, `data_io.py`, `contaminate.py`, `dataset.py`, `model.py`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Arquivos recebidos:', list(uploaded.keys()))

import pathlib
assert pathlib.Path('best_model.keras').exists(), \
    'ERRO: best_model.keras não encontrado. Inclua-o no upload.'
print('best_model.keras encontrado — OK')

---
## 5 · Montar estrutura de pacotes

In [ ]:
import shutil, pathlib

for folder in ['src', 'src/noise', 'src/models']:
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)
    pathlib.Path(f'{folder}/__init__.py').touch()

moves = {
    'data_io.py'     : 'src/data_io.py',
    'contaminate.py' : 'src/noise/contaminate.py',
    'dataset.py'     : 'src/models/dataset.py',
    'model.py'       : 'src/models/model.py',
}
for src_file, dst in moves.items():
    if pathlib.Path(src_file).exists():
        shutil.copy(src_file, dst)
        print(f'  {src_file}  →  {dst}')
    else:
        print(f'  AVISO: {src_file} não encontrado.')

!find src -name '*.py' | sort

---
## 6 · Carregar o modelo
Testa forma de entrada/saída com um batch fictício.

In [ ]:
MODEL_PATH = 'best_model.keras'
import sys; sys.path.insert(0, '/content')

print(f'Carregando {MODEL_PATH}...')
model = tf.keras.models.load_model(MODEL_PATH)
print('Modelo carregado com sucesso!')
print(f'Parâmetros treináveis: {model.count_params():,}')

x_dummy = np.random.randn(4, 1024, 1).astype('float32')
y_dummy = model(x_dummy, training=False)
assert y_dummy.shape == x_dummy.shape, f'Forma inesperada: {y_dummy.shape}'
print(f'\nSanity check OK — {x_dummy.shape} → {y_dummy.shape}')

---
## 7 · Baixar dados do PhysioNet
⏱ **5–10 minutos.** Apenas uma vez por sessão.

In [ ]:
from src.data_io import ensure_datasets, ALL_RECORDS, NOISE_RECORDS

print(f'Baixando {len(ALL_RECORDS)} registros MIT-BIH + {len(NOISE_RECORDS)} arquivos de ruído...')
ensure_datasets()
print('\nDados prontos.')

---
## 8 · Construir tabela de teste
Gera pares (ruidoso, limpo) determinísticos — grade `ruído × SNR` com os 10 registros de teste.

In [ ]:
from src.models.dataset import build_data_pools, build_test_table

print('Construindo pools de dados...')
pools = build_data_pools()
print(f'  janelas treino : {pools.ecg_train_windows.shape}')
print(f'  janelas val    : {pools.ecg_val_windows.shape}')
print(f'  janelas teste  : {pools.ecg_test_windows.shape}')

print('\nGerando tabela de teste (seed fixa = 2026)...')
test_table = build_test_table(pools)

noise_types = sorted(set(k[0] for k in test_table))
snr_levels  = sorted(set(k[1] for k in test_table))
print(f'  Tipos de ruído : {noise_types}')
print(f'  Níveis de SNR  : {snr_levels} dB')
print(f'  Combinações    : {len(test_table)}')

X_ex, Y_ex = test_table[(noise_types[0], snr_levels[0])]
print(f'  Forma por célula: X={X_ex.shape}  Y={Y_ex.shape}')

---
## 9 · Métricas quantitativas (MAE, MSE, SNR, ΔSNR)

Para cada combinação (ruído, SNR):
- **MAE** — erro absoluto médio
- **MSE** — erro quadrático médio
- **SNR out (dB)** — SNR do sinal filtrado vs. referência
- **ΔSNR (dB)** — melhora de SNR entregue pelo modelo

In [ ]:
import pandas as pd

def compute_snr(signal, noise_component):
    p_sig   = np.mean(signal ** 2, axis=(1, 2))
    p_noise = np.mean(noise_component ** 2, axis=(1, 2))
    p_noise = np.where(p_noise == 0, 1e-12, p_noise)
    return 10 * np.log10(p_sig / p_noise)

rows = []
for (noise_type, snr_in), (X, Y) in test_table.items():
    Y_hat = model.predict(X, verbose=0)

    residual_out = Y_hat - Y
    mae     = float(np.mean(np.abs(Y_hat - Y)))
    mse     = float(np.mean((Y_hat - Y) ** 2))
    snr_out = float(np.mean(compute_snr(Y, residual_out)))
    delta   = snr_out - snr_in

    rows.append(dict(noise=noise_type, snr_in=snr_in,
                     mae=mae, mse=mse, snr_out=snr_out, delta_snr=delta))
    print(f'  {noise_type:>5s}  SNR_in={snr_in:+3d} dB  '
          f'MAE={mae:.4f}  MSE={mse:.4f}  '
          f'SNR_out={snr_out:+.1f} dB  ΔSNR={delta:+.1f} dB')

results_df = pd.DataFrame(rows)
print('\n--- Resumo por tipo de ruído ---')
print(results_df.groupby('noise')[['mae','mse','snr_out','delta_snr']].mean().round(4))

---
## 10 · Heatmap ΔSNR

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

pivot = results_df.pivot(index='noise', columns='snr_in', values='delta_snr')

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
               norm=mcolors.TwoSlopeNorm(vcenter=0,
                                          vmin=pivot.values.min(),
                                          vmax=pivot.values.max()))
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{v:+d} dB' for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('SNR de entrada')
ax.set_ylabel('Tipo de ruído')
ax.set_title('ΔSNR entregue pelo modelo (dB)  [verde = melhora, vermelho = piora]')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i, j]:+.1f}', ha='center', va='center',
                fontsize=11, fontweight='bold')

plt.colorbar(im, ax=ax, label='ΔSNR (dB)')
plt.tight_layout()
try:
    path = f'{DRIVE_DIR}/validacao/heatmap_delta_snr.png'
    plt.savefig(path, dpi=150)
    print('Salvo em:', path)
except: pass
plt.show()

---
## 11 · Visualização: ruidoso × filtrado × referência

In [ ]:
NOISE_VIS = 'bw'  # bw | ma | em | 60hz | 50hz
SNR_VIS   = 6     # 0 | 6 | 12 | 18
N_JANELAS = 3

X_vis, Y_vis = test_table[(NOISE_VIS, SNR_VIS)]
Y_hat_vis    = model.predict(X_vis[:N_JANELAS], verbose=0)

t = np.arange(1024) / 360

fig, axes = plt.subplots(N_JANELAS, 3, figsize=(16, 3.5 * N_JANELAS), sharex=True)
if N_JANELAS == 1:
    axes = axes[np.newaxis, :]

for i in range(N_JANELAS):
    mae_i = np.mean(np.abs(Y_hat_vis[i] - Y_vis[i]))
    axes[i, 0].plot(t, X_vis[i, :, 0],     color='tab:orange', lw=0.8)
    axes[i, 0].set_title(f'Janela {i} · Ruidoso  ({NOISE_VIS}, SNR={SNR_VIS} dB)')
    axes[i, 1].plot(t, Y_hat_vis[i, :, 0], color='tab:blue',   lw=0.8)
    axes[i, 1].set_title(f'Janela {i} · Filtrado  (MAE={mae_i:.4f})')
    axes[i, 2].plot(t, Y_vis[i, :, 0],     color='tab:green',  lw=0.8)
    axes[i, 2].set_title(f'Janela {i} · Referência limpa')
    for ax in axes[i]:
        ax.set_ylabel('mV (norm.)')
        ax.grid(True, alpha=0.3)

axes[-1, 1].set_xlabel('Tempo (s)')
plt.suptitle(f'Filtragem — {NOISE_VIS.upper()}, SNR={SNR_VIS} dB', fontsize=13, y=1.01)
plt.tight_layout()
try:
    path = f'{DRIVE_DIR}/validacao/filtragem_{NOISE_VIS}_{SNR_VIS}dB.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print('Salvo em:', path)
except: pass
plt.show()

---
## 12 · Curva MAE × SNR de entrada por tipo de ruído

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for noise_type in results_df['noise'].unique():
    sub = results_df[results_df['noise'] == noise_type].sort_values('snr_in')
    axes[0].plot(sub['snr_in'], sub['mae'],       marker='o', label=noise_type)
    axes[1].plot(sub['snr_in'], sub['delta_snr'],  marker='o', label=noise_type)

axes[0].set_title('MAE vs SNR de entrada')
axes[0].set_xlabel('SNR de entrada (dB)')
axes[0].set_ylabel('MAE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color='gray', lw=0.8, linestyle='--')
axes[1].set_title('ΔSNR vs SNR de entrada')
axes[1].set_xlabel('SNR de entrada (dB)')
axes[1].set_ylabel('ΔSNR (dB)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
try:
    path = f'{DRIVE_DIR}/validacao/curvas_mae_snr.png'
    plt.savefig(path, dpi=150)
    print('Salvo em:', path)
except: pass
plt.show()